# DSPy Muay Thai Strike Classifier

This notebook implements a DSPy-based system using BootstrapFewShot optimizer to help Google Gemini classify Muay Thai strike combinations in videos.


## Section 1: Setup and Configuration


In [1]:
import dspy
import google.generativeai as genai
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
from typing import List, Dict, Tuple

# Load environment variables
load_dotenv()

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in .env file.")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Configure DSPy with Gemini 2.5 Flash
# Try gemini-2.0-flash-exp first, fallback to gemini-1.5-flash if needed
try:
    lm = dspy.LM(model="gemini/gemini-2.5-flash", api_key=GEMINI_API_KEY)
    print('using gemini-2.5-flash')
except:
    print("Error: Failed to configure Gemini 2.5 Flash. Please check your API key and model availability.")

dspy.configure(lm=lm)

# Data paths
DATA_ROOT = Path("data/combos")
SHADOW_DIR = DATA_ROOT / "shadow"
HEAVY_BAG_DIR = DATA_ROOT / "heavy_bag"

# Validation split size
VAL_SPLIT_SIZE = 7

print("Setup complete!")
print(f"Data directories: {SHADOW_DIR}, {HEAVY_BAG_DIR}")


using gemini-2.5-flash
Setup complete!
Data directories: data\combos\shadow, data\combos\heavy_bag


## Section 2: Data Loading


In [2]:
def load_labels(json_path: Path) -> Dict[str, List[str]]:
    """Load true labels from JSON file."""
    with open(json_path, 'r') as f:
        return json.load(f)

def create_examples(labels: Dict[str, List[str]], video_dir: Path, dataset_name: str) -> List[dspy.Example]:
    """Create DSPy examples from labels and video directory."""
    examples = []
    for video_id, strikes in labels.items():
        video_path = video_dir / f"{video_id}.mov"
        if video_path.exists():
            example = dspy.Example(
                video_path=str(video_path),
                strikes=strikes
            ).with_inputs("video_path")
            examples.append(example)
        else:
            print(f"Warning: Video {video_path} not found")
    return examples

# Load labels from both datasets
shadow_labels = load_labels(SHADOW_DIR / "true_labels.json")
heavy_bag_labels = load_labels(HEAVY_BAG_DIR / "true_labels.json")

# Create examples
shadow_examples = create_examples(shadow_labels, SHADOW_DIR, "shadow")
heavy_bag_examples = create_examples(heavy_bag_labels, HEAVY_BAG_DIR, "heavy_bag")

print(f"Loaded {len(shadow_examples)} shadow examples")
print(f"Loaded {len(heavy_bag_examples)} heavy_bag examples")


Loaded 20 shadow examples
Loaded 19 heavy_bag examples


In [3]:
# Split data into train and validation
# Randomly select VAL_SPLIT_SIZE from each dataset for training (flipped for DSPy recommendation)
# This gives us a smaller training set and larger validation set to prevent overfitting

random.seed(42)  # For reproducibility

# Shadow split
shadow_indices = list(range(len(shadow_examples)))
random.shuffle(shadow_indices)
shadow_train_indices = set(shadow_indices[:VAL_SPLIT_SIZE])
shadow_train = [shadow_examples[i] for i in shadow_train_indices]
shadow_val = [shadow_examples[i] for i in range(len(shadow_examples)) if i not in shadow_train_indices]

# Heavy bag split
heavy_bag_indices = list(range(len(heavy_bag_examples)))
random.shuffle(heavy_bag_indices)
heavy_bag_train_indices = set(heavy_bag_indices[:VAL_SPLIT_SIZE])
heavy_bag_train = [heavy_bag_examples[i] for i in heavy_bag_train_indices]
heavy_bag_val = [heavy_bag_examples[i] for i in range(len(heavy_bag_examples)) if i not in heavy_bag_train_indices]

# Combine train and validation sets
train_examples = shadow_train + heavy_bag_train
val_examples = shadow_val + heavy_bag_val

print(f"Training set: {len(train_examples)} examples ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation set: {len(val_examples)} examples ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")

# Display some statistics
all_strikes = set()
for ex in train_examples + val_examples:
    all_strikes.update(ex.strikes)

print(f"\nUnique strike types: {sorted(all_strikes)}")
print(f"Total unique strike types: {len(all_strikes)}")


Training set: 14 examples (7 shadow + 7 heavy_bag)
Validation set: 25 examples (13 shadow + 12 heavy_bag)

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'lowkick', 'rear_elbow', 'rear_hook', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 13


## Section 3: DSPy Signature and Module


In [4]:
class StrikeClassifier(dspy.Signature):
    """Classify Muay Thai strikes in a video sequence. Watch the video and identify all strikes in the order they appear. Return a JSON array of strike names."""
    video_path: str = dspy.InputField(desc="Path to video file")
    strikes: str = dspy.OutputField(desc="JSON array of strike names in sequence, e.g. [\"jab\", \"cross\", \"lead_hook\"]")


In [5]:
class MuayThaiClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classifier = dspy.ChainOfThought(StrikeClassifier)
    
    def forward(self, video_path: str):
        return self.classifier(video_path=video_path)

# Initialize classifier
classifier = MuayThaiClassifier()

print("Classifier module created!")


Classifier module created!


In [6]:
# Test basic classification on a sample video before optimization
# Note: This requires Gemini to process video, which may take some time

test_example = train_examples[0]
print(f"Testing on: {test_example.video_path}")
print(f"True labels: {test_example.strikes}")

try:
    result = classifier(video_path=test_example.video_path)
    print(f"Predicted: {result.strikes}")
    
    # Try to parse JSON
    import json
    try:
        predicted_strikes = json.loads(result.strikes)
        print(f"Parsed strikes: {predicted_strikes}")
    except json.JSONDecodeError:
        print("Warning: Could not parse JSON from response")
except Exception as e:
    print(f"Error during classification: {e}")
    print("This is expected if Gemini video API requires different setup. Continuing with optimizer...")


Testing on: data\combos\shadow\5.mov
True labels: ['jab', 'cross', 'lead_hook', 'lead_hook']
Predicted: ["jab", "cross", "lead_kick", "cross", "jab", "rear_kick"]
Parsed strikes: ['jab', 'cross', 'lead_kick', 'cross', 'jab', 'rear_kick']


## Section 4: BootstrapFewShot Optimizer


In [7]:
def exact_match_metric(gold, pred, trace=None):
    """Evaluate exact sequence match between predicted and true strikes."""
    # Parse predicted strikes if it's a JSON string
    try:
        if isinstance(pred.strikes, str):
            predicted_strikes = json.loads(pred.strikes)
        else:
            predicted_strikes = pred.strikes
    except (json.JSONDecodeError, AttributeError):
        # If parsing fails, try to extract from string
        predicted_strikes = []
        if hasattr(pred, 'strikes'):
            pred_str = str(pred.strikes)
            # Try to find JSON array in the string
            import re
            match = re.search(r'\[.*?\]', pred_str)
            if match:
                try:
                    predicted_strikes = json.loads(match.group())
                except:
                    predicted_strikes = []
    
    # Get true strikes
    true_strikes = gold.strikes if hasattr(gold, 'strikes') else gold
    
    # Check exact match
    if predicted_strikes == true_strikes:
        return True
    
    return False

def sequence_accuracy_metric(gold, pred, trace=None):
    """Calculate sequence accuracy (partial credit for correct strikes in order)."""
    try:
        if isinstance(pred.strikes, str):
            predicted_strikes = json.loads(pred.strikes)
        else:
            predicted_strikes = pred.strikes
    except (json.JSONDecodeError, AttributeError):
        predicted_strikes = []
        if hasattr(pred, 'strikes'):
            pred_str = str(pred.strikes)
            import re
            match = re.search(r'\[.*?\]', pred_str)
            if match:
                try:
                    predicted_strikes = json.loads(match.group())
                except:
                    predicted_strikes = []
    
    true_strikes = gold.strikes if hasattr(gold, 'strikes') else gold
    
    if not predicted_strikes or not true_strikes:
        return 0.0
    
    # Calculate longest common subsequence ratio
    min_len = min(len(predicted_strikes), len(true_strikes))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if predicted_strikes[i] == true_strikes[i])
    return matches / len(true_strikes)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [8]:
# Configure BootstrapFewShot optimizer
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=exact_match_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8
)

print("Optimizer configured!")
print(f"Max bootstrapped demos: 4")
print(f"Max labeled demos: 8")


Optimizer configured!
Max bootstrapped demos: 4
Max labeled demos: 8


In [9]:
# Run optimization on training set
# This may take a while as it needs to process videos through Gemini API

print("Starting optimization...")
print(f"Training on {len(train_examples)} examples")

try:
    optimized_classifier = optimizer.compile(
        student=classifier,
        trainset=train_examples
    )
    print("\nOptimization complete!")
    # print(f"Optimized classifier has {len(optimized_classifier)} compiled demonstrations")
except Exception as e:
    print(f"Error during optimization: {e}")
    print("This may be due to API rate limits or video processing issues.")
    print("You may need to adjust the optimizer parameters or reduce the training set size.")
    raise


Starting optimization...
Training on 14 examples


100%|██████████| 14/14 [00:00<00:00, 33.65it/s]

Bootstrapped 0 full traces after 13 examples for up to 1 rounds, amounting to 14 attempts.

Optimization complete!


In [10]:
# Display selected examples
if hasattr(optimized_classifier.classifier, '_compiled_demos'):
    print("Selected few-shot examples:")
    for i, demo in enumerate(optimized_classifier.classifier._compiled_demos[:5], 1):
        print(f"\nExample {i}:")
        print(f"  Video: {demo.video_path}")
        print(f"  Strikes: {demo.strikes}")


## Section 5: Evaluation


In [11]:
# Evaluate optimized model on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")

results = []
exact_matches = 0
total_accuracy = 0.0

for i, example in enumerate(val_examples):
    try:
        prediction = optimized_classifier(video_path=example.video_path)
        
        # Parse prediction
        try:
            if isinstance(prediction.strikes, str):
                predicted_strikes = json.loads(prediction.strikes)
            else:
                predicted_strikes = prediction.strikes
        except (json.JSONDecodeError, AttributeError):
            predicted_strikes = []
            if hasattr(prediction, 'strikes'):
                pred_str = str(prediction.strikes)
                import re
                match = re.search(r'\[.*?\]', pred_str)
                if match:
                    try:
                        predicted_strikes = json.loads(match.group())
                    except:
                        predicted_strikes = []
        
        true_strikes = example.strikes
        exact_match = (predicted_strikes == true_strikes)
        accuracy = sequence_accuracy_metric(example, prediction)
        
        if exact_match:
            exact_matches += 1
        total_accuracy += accuracy
        
        results.append({
            'video_path': example.video_path,
            'true_strikes': true_strikes,
            'predicted_strikes': predicted_strikes,
            'exact_match': exact_match,
            'sequence_accuracy': accuracy
        })
        
        if (i + 1) % 5 == 0:
            print(f"Processed {i + 1}/{len(val_examples)} examples...")
            
    except Exception as e:
        print(f"Error processing {example.video_path}: {e}")
        results.append({
            'video_path': example.video_path,
            'true_strikes': example.strikes,
            'predicted_strikes': [],
            'exact_match': False,
            'sequence_accuracy': 0.0,
            'error': str(e)
        })

print("\nEvaluation complete!")


Evaluating on 25 validation examples...
Processed 5/25 examples...
Processed 10/25 examples...
Processed 15/25 examples...
Processed 20/25 examples...
Processed 25/25 examples...

Evaluation complete!


In [12]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0
average_sequence_accuracy = total_accuracy / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print(f"Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strikes']}")
    print(f"  Pred:  {result['predicted_strikes']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'} (Accuracy: {result['sequence_accuracy']:.2%})")


EVALUATION RESULTS
Exact Match Accuracy: 12.00% (3/25)
Average Sequence Accuracy: 39.93%

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  ['jab', 'cross', 'lead_hook', 'roundhouse']
  Pred:  ['jab', 'cross', 'lead_hook', 'roundhouse']
  Match: ✓ (Accuracy: 100.00%)

Example 2:
  Video: 2.mov
  True:  ['jab', 'cross', 'lowkick']
  Pred:  ['jab', 'cross', 'lead_hook', 'roundhouse']
  Match: ✗ (Accuracy: 66.67%)

Example 3:
  Video: 3.mov
  True:  ['lead_uppercut', 'cross', 'rear_elbow', 'knee']
  Pred:  ['jab', 'cross', 'roundhouse']
  Match: ✗ (Accuracy: 25.00%)

Example 4:
  Video: 4.mov
  True:  ['jab', 'cross', 'lead_hook', 'lead_hook', 'roundhouse']
  Pred:  ['jab', 'cross', 'roundhouse']
  Match: ✗ (Accuracy: 40.00%)

Example 5:
  Video: 7.mov
  True:  ['lead_uppercut', 'cross', 'lead_hook']
  Pred:  ['jab', 'cross', 'switch_kick']
  Match: ✗ (Accuracy: 33.33%)


In [13]:
# Calculate per-strike accuracy
from collections import defaultdict

strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strikes = result['true_strikes']
    pred_strikes = result['predicted_strikes']
    
    # Count true strikes
    for strike in true_strikes:
        strike_stats[strike]['total'] += 1
    
    # Count predicted strikes
    for strike in pred_strikes:
        strike_stats[strike]['predicted'] += 1
    
    # Count correct predictions at each position
    min_len = min(len(true_strikes), len(pred_strikes))
    for i in range(min_len):
        if true_strikes[i] == pred_strikes[i]:
            strike_stats[true_strikes[i]]['correct'] += 1

print("\nPer-Strike Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike Accuracy:
------------------------------------------------------------
  cross               : 75.00% (15/20 correct, 26 predicted)
  jab                 : 60.87% (14/23 correct, 18 predicted)
  knee                :  0.00% (0/4 correct, 1 predicted)
  lead_elbow          :  0.00% (0/1 correct, 1 predicted)
  lead_hook           : 18.75% (3/16 correct, 13 predicted)
  lead_uppercut       :  0.00% (0/4 correct, 0 predicted)
  lowkick             :  0.00% (0/1 correct, 0 predicted)
  rear_elbow          :  0.00% (0/2 correct, 0 predicted)
  rear_hook           :  0.00% (0/2 correct, 0 predicted)
  rear_uppercut       :  0.00% (0/0 correct, 1 predicted)
  roundhouse          : 25.00% (3/12 correct, 23 predicted)
  switch_kick         :  0.00% (0/1 correct, 2 predicted)
  switch_knee         :  0.00% (0/1 correct, 0 predicted)
  teep                :  0.00% (0/3 correct, 2 predicted)


## Section 6: Results and Analysis


In [14]:
# Save optimized prompts and results
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'average_sequence_accuracy': average_sequence_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "combos_dspy.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'combos_dspy.json'}")


Results saved to results\combos_dspy.json


In [15]:
# Display summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash")
print(f"Optimizer: BootstrapFewShot")
print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"  Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"\nResults saved to: {output_dir}/")
print("=" * 60)

optimized_classifier.save("optimized_classifier.pkl")



SUMMARY
Model: Gemini 2.5 Flash
Optimizer: BootstrapFewShot
Training examples: 14
Validation examples: 25

Final Performance:
  Exact Match Accuracy: 12.00%
  Average Sequence Accuracy: 39.93%

Results saved to: results/
